In [1]:
for name in dir():
    if not name.startswith("_") and name not in ["In", "Out", "get_ipython", "exit", "quit"]:
        del globals()[name]


---
---

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [3]:
# import pandas as pd
# data = pd.read_csv(r"C:\Users\USER\Downloads\Churn_Modelling.csv").drop(columns=['RowNumber', 'CustomerId', 'Surname'])
# data = pd.get_dummies(data, columns=['Geography', 'Gender'], drop_first=True, dtype=int)

In [4]:
# exited_1 = data[data['Exited']==1].reset_index(drop=True)
# exited_0 = data[data['Exited']==0].sample(n=2963, random_state=42).reset_index(drop=True)
# data_5000 = pd.concat([exited_0, exited_1], axis=0)

In [5]:
# data = data_5000.drop(columns=['Exited']) #要跑GAP, no_Y就這樣存
# data_5000.to_csv('churn.csv', sep='\t', encoding='utf-8', index=False) #整筆這樣存, 後續再改data儲存後跑GAP才不會有問題

---
---

In [6]:
churn = pd.read_csv(r"C:\Users\User\Documents\GitHub\psychic-spoon\DataSet\churn.csv", sep='\t')
churn['filename'] = [f'{i}' for i in range(1, 5001)]
churn['filename'] = churn['filename'].astype(int)
# banknote = banknote.drop(columns=['class'])
# churn.to_csv('churn.csv', sep='\t', encoding='utf-8', index=False) 
gap_churn_order = pd.read_csv(r"C:\Users\User\Documents\GitHub\psychic-spoon\DataSet\gap_churn_歐式.txt", sep='\t')
raw = '8  1  0  8  1  0  6  7  7  1  2  6  0  1  3  3  6  7  7  1  0  0  0  7  0  7  0  6  3  6  7  1  0  3  1  4  1  0  1  7  3  7  7  4  3  4  8  1  3  0  0  1  1  6  7  4  0  1  8  1  0  7  1  0  7  6  1  1  7  1  1  4  6  5  0  3  0  0  6  7  0  4  0  1  4  1  1  6  6  0  0  0  1  1  8  0  1  1  6  2  6  3  3  4  0  3  3  7  3  1  8  7  7  7  0  2  8  3  7  6  4  8  0  1  7  8  8  6  6  1  0  3  6  0  6  7  1  6  5  4  7  6  1  1  4  7  1  0  2  1  1  3  0  3  7  6  2  3  3  7  1  1  4  1  1  6  3  3  8  3  1  6  6  7  0  0  6  3  6  6  6  8  7  8  6  8  7  0  0  6  1  1  0  6  0  7  7  7  7  4  7  0  1  8  5  8  6  0  4  3  0  3  1  0  0  3  4  0  1  8  6  1  0  7  6  1  6  3  3  3  3  4  4  3  7  3  3  6  0  1  8  8  8  7  3  3  7  3  3  7  8  1  1  3  3  7  1  1  2  8  4  0  3  8  7  8  0  3  7  7  3  7  6  6  0  8  0  8  6  1  3  7  3  8  8  7  1  6  0  0  1  8  2  3  1  0  1  1  0  7  1  7  1  1  6  1  5  3  4  0  0  6  1  6  0  3  1  6  7  6  3  3  0  0  7  0  7  6  3  1  6  7  6  7  6  1  0  0  6  7  3  2  0  0  0  7  8  3  6  1  6  6  3  8  8  7  8  5  1  1  3  6  6  1  3  3  7  7  5  1  0  7  6  0  6  5  6  7  1  8  8  8  0  6  0  7  0  0  1  4  0  7  4  7  3  8  0  3  8  8  1  0  8  3  1  3  7  6  8  8  1  1  1  1  4  2  7  0  0  0  6  6  3  7  1  7  1  1  4  1  0  0  6  7  1  0  0  3  3  6  1  7  3  1  6  3  1  7  0  0  3  7  6  0  0  0  7  1  3  0  1  3  1  0  3  7  4  6  7  3  3  0  2  3  3  4  1  2  7  1  1  8  1  6  0  7  1  4  0  0  0  1  1  3  6  2  8  3  6  6  0  8  3  0  8  2  8  0  0  6  1  6  3  3  0  8  1  0  3  3  0  7  0  8  3  7  0  6  7  8  4  3  1  7  7  1  6  8  7  7  3  1  0  2  7  1  1  0  0  1  7  1  7  7  4  1  6  4  3  7  3  6  3  7  0  3  0  0  1  7  7  3  3  0  7  1  4  8  7  0  6  6  7  7  8  7  7  1  1  4  0  8  4  1  7  3  7  7  3  8  3  1  0  6  0  3  5  0  0  1  1  4  4  4  6  0  6  7  3  7  0  8  7  1  0  7  4  1  8  8  0  0  8  3  1  1  8  8  7  1  1  1  0  3  0  1  6  0  2  6  3  7  3  7  8  0  1  1  7  7  0  3  3  2  7  1  0  3  3  4  1  0  7  3  8  1  3  0  7  2  0  0  3  2  6  4  1  1  8  2  7  1  1  3  0  3  7  0  0  7  6  0  7  0  4  6  0  8  3  8  8  1  1  8  0  1  7  0  0  8  7  3  7  1  8  7  0  3  1  8  1  0  1  3  8  0  4  7  7  8  3  3  0  1  0  0  8  7  6  1  3  7  3  3  6  4  0  6  7  8  1  4  7  4  5  0  8  3  7  1  1  3  0  4  0  0  0  3  1  3  0  0  7  8  8  3  1  1  6  6  6  6  1  0  1  7  7  3  3  0  7  2  3  1  7  6  8  0  8  8  6  7  1  1  7  6  6  1  0  3  8  0  3  6  8  7  4  2  6  0  2  7  6  6  3  0  0  7  1  1  1  0  0  1  8  2  1  1  6  2  6  7  0  7  0  1  0  0  8  7  0  6  0  7  1  1  8  7  3  4  3  3  6  3  1  7  1  8  8  3  0  7  7  0  7  0  6  0  6  0  3  7  3  1  3  1  1  3  6  1  7  5  7  3  7  7  1  1  1  6  0  6  0  3  0  4  4  0  0  6  1  3  6  8  1  3  4  1  7  6  0  7  8  1  0  7  0  7  8  6  0  7  1  8  1  8  1  7  8  6  0  6  6  8  6  3  5  7  0  1  1  6  3  1  1  3  1  1  8  1  8  7  0  7  0  3  6  7  0  0  6  0  3  7  3  1  7  1  8  8  3  0  1  0  0  0  6  6  7  3  1  6  4  7  3  6  7  0  7  2  0  7  3  4  3  7  7  0  6  8  1  1  1  1  8  1  0  3  1  8  6  1  3  6  1  6  0  3  3  3  4  6  0  3  0  8  1  1  7  0  1  6  0  3  4  6  0  3  7  0  3  8  7  6  8  0  0  3  8  3  0  7  6  0  4  3  0  6  5  8  0  7  0  1  0  3  6  7  1  1  7  0  3  1  7  0  3  1  0  1  3  0  7  3  1  6  3  8  8  0  3  6  3  7  0  3  8  6  6  7  3  3  7  0  0  5  4  7  7  3  6  3  7  1  3  3  2  3  3  3  0  1  0  1  0  1  1  0  1  3  8  6  8  7  3  1  3  3  3  7  0  7  7  0  7  4  3  1  0  7  1  6  4  3  7  0  0  0  0  2  0  0  1  4  7  7  1  7  6  4  1  8  4  0  1  3  1  3  6  3  5  0  1  0  1  4  7  7  1  1  0  7  7  1  3  6  1  0  0  0  0  0  8  6  1  1  7  8  0  1  3  8  8  4  0  0  3  7  3  4  1  7  6  0  7  7  7  3  3  0  0  3  8  8  8  0  1  3  1  6  1  3  3  3  3  7  3  1  7  3  6  1  6  8  3  3  6  1  7  0  7  2  3  7  4  1  3  8  0  6  0  3  7  7  2  0  1  6  3  4  0  7  1  1  4  0  8  1  8  0  3  1  0  0  0  6  7  8  0  6  0  7  3  0  0  7  1  6  3  3  8  4  2  1  7  0  5  0  1  8  6  4  6  7  0  6  0  0  0  3  1  1  7  3  6  3  6  1  0  7  3  0  7  0  0  3  8  7  3  6  3  3  1  7  1  1  3  7  7  1  8  8  1  1  6  7  6  1  1  3  0  3  1  7  7  1  7  1  4  0  8  1  7  6  7  6  4  3  6  6  0  7  6  6  8  1  0  6  0  7  1  7  0  1  8  7  8  3  6  1  3  6  6  7  3  7  6  1  0  1  1  1  7  6  1  1  7  2  8  3  7  6  7  1  3  8  0  0  1  7  1  1  0  6  0  8  1  5  8  1  2  0  6  0  3  7  0  4  0  1  0  1  4  7  0  0  3  5  3  6  0  2  1  0  3  1  7  2  3  1  8  0  6  0  6  0  0  1  4  8  0  0  6  0  6  0  6  1  7  8  3  0  8  3  3  0  0  1  6  1  7  5  3  1  6  3  1  4  1  0  7  3  1  0  7  0  7  6  0  0  1  0  6  8  7  8  1  8  4  7  0  1  3  7  6  2  6  6  8  1  0  3  5  2  8  3  0  8  7  3  0  8  7  8  3  4  6  4  0  6  1  7  0  7  7  1  6  3  1  0  1  1  2  1  0  0  7  7  0  1  6  7  7  7  0  8  7  1  7  1  0  1  8  3  7  0  7  1  1  7  7  8  7  3  6  0  1  6  0  7  1  0  1  8  7  7  2  1  7  1  0  0  4  6  6  7  0  8  8  8  0  1  3  3  1  0  4  0  7  3  0  4  8  0  1  3  1  6  4  3  4  0  0  1  0  0  1  0  1  7  7  3  8  1  3  4  6  0  1  8  4  6  1  0  1  3  0  3  0  0  1  6  0  7  6  1  7  0  1  1  1  3  5  1  3  1  1  3  1  0  3  0  8  7  7  6  4  1  1  4  3  3  8  7  0  0  7  1  4  7  3  0  6  4  3  6  1  1  4  3  2  3  1  1  3  1  6  7  3  6  6  0  6  0  6  5  8  8  0  7  3  3  7  4  4  7  3  7  3  7  1  0  7  3  4  6  8  3  7  8  3  3  7  0  0  3  3  1  3  0  2  3  1  6  7  0  7  6  3  0  7  4  2  0  1  0  0  3  5  0  6  4  0  6  2  7  1  1  0  0  1  6  8  0  8  8  6  1  0  0  7  7  0  1  3  7  8  2  1  1  3  0  1  7  2  8  3  1  3  0  1  1  7  6  6  3  1  0  3  0  3  1  1  0  3  8  7  3  7  8  3  3  0  7  7  1  3  7  3  1  0  3  5  7  3  1  1  5  3  0  0  6  7  0  6  0  7  6  0  0  1  0  3  7  3  0  7  1  6  0  7  3  7  1  6  6  6  1  1  1  7  7  3  7  3  8  0  3  1  3  1  0  7  1  1  6  6  7  1  3  7  0  8  0  7  0  1  0  5  1  1  1  8  2  0  1  1  0  6  6  7  8  6  6  1  7  1  7  8  6  8  1  0  8  6  2  3  6  3  3  0  1  7  7  7  0  4  3  0  7  1  1  6  7  2  5  1  0  8  7  1  3  3  4  8  1  6  4  6  6  7  6  6  0  7  7  7  1  0  0  1  8  1  1  0  3  1  1  7  3  1  3  4  6  0  8  1  7  1  3  3  3  1  5  3  1  7  7  6  3  3  0  1  1  1  3  7  2  7  7  0  8  6  1  1  3  1  3  7  1  6  8  7  0  1  0  7  7  7  7  6  6  1  0  3  7  4  1  7  0  4  7  6  1  1  0  8  1  1  1  7  3  1  8  1  3  0  7  1  0  1  6  0  1  6  3  4  0  8  4  1  6  7  7  0  1  7  0  1  3  6  7  3  1  6  4  1  1  1  3  3  4  0  8  7  4  1  2  4  1  1  1  8  1  6  4  1  1  3  0  0  0  7  1  0  8  0  4  7  7  0  7  3  1  0  6  0  0  1  5  4  3  3  1  6  7  0  8  4  8  6  3  8  1  0  8  8  1  8  7  7  8  0  2  1  0  6  3  1  7  6  3  6  1  1  3  4  8  1  0  1  1  7  0  0  1  6  6  6  1  6  7  6  7  0  0  0  7  2  3  8  3  1  7  8  6  3  7  7  7  1  7  7  3  1  8  7  8  0  3  7  6  0  6  3  8  3  7  1  7  1  6  0  1  8  3  7  7  3  4  7  0  3  8  0  3  3  3  1  2  3  0  6  8  0  4  0  0  6  7  1  3  0  3  2  3  3  3  1  1  3  0  0  1  3  7  7  1  3  4  8  4  1  0  3  3  7  1  3  3  7  3  0  1  3  0  5  6  7  1  1  1  3  1  0  7  0  1  0  7  0  1  3  1  7  0  6  1  1  0  1  1  0  8  7  0  0  7  7  7  0  5  0  1  6  1  6  0  0  1  0  7  1  6  6  1  6  0  0  4  1  6  1  0  5  1  1  6  3  3  8  1  0  2  3  0  0  0  6  7  7  2  7  1  1  3  4  3  8  6  8  1  7  4  1  0  1  0  7  0  0  1  1  3  1  0  3  0  0  7  3  6  1  6  3  8  7  0  0  3  1  1  1  7  0  0  0  6  7  6  3  0  7  1  6  7  4  3  1  8  6  1  1  7  8  6  0  0  1  1  8  2  7  0  1  0  1  0  0  1  3  0  1  2  7  7  3  8  7  0  1  1  1  1  1  3  0  6  3  1  1  1  6  7  0  6  8  8  6  3  3  1  1  0  3  3  0  1  8  2  6  7  4  3  1  4  0  8  8  4  8  7  1  1  3  0  6  8  7  8  1  8  7  0  8  0  7  4  0  1  7  6  3  4  0  3  7  8  3  8  2  3  7  6  0  7  0  1  8  7  3  3  1  0  4  8  8  1  6  6  7  0  3  8  7  1  7  8  1  7  8  1  0  1  7  0  6  6  0  1  0  0  1  8  0  0  3  0  7  3  0  1  8  6  7  8  1  7  1  0  6  0  8  1  6  0  1  0  0  1  7  4  7  0  1  0  4  1  7  6  6  4  6  1  3  1  1  6  0  3  0  1  0  1  7  3  6  3  4  2  5  1  6  6  6  7  3  1  6  0  0  0  1  2  7  1  6  0  7  4  6  7  3  8  8  0  4  2  8  8  6  7  1  3  4  8  7  7  6  4  3  7  6  2  8  3  1  8  6  6  3  6  1  8  1  3  7  1  0  0  0  7  6  8  3  3  0  0  1  1  0  3  8  3  0  0  6  0  1  7  0  3  3  3  3  1  0  0  0  1  3  4  5  7  1  3  1  0  0  1  8  6  6  1  0  1  3  1  7  1  3  6  3  1  0  1  3  1  1  7  7  3  8  0  1  0  7  3  8  3  6  4  3  0  6  6  3  1  6  0  3  6  4  6  3  1  7  8  0  8  8  8  0  5  4  7  3  3  7  6  6  3  4  7  6  3  2  0  0  0  3  1  7  6  1  6  5  7  6  1  0  8  6  4  0  3  7  3  1  7  1  1  8  0  7  0  0  6  4  4  7  0  3  1  8  0  7  0  0  3  0  7  8  1  1  3  8  3  8  0  1  8  0  3  7  7  1  0  1  0  3  8  7  0  7  1  7  4  1  6  8  1  1  3  6  0  1  3  1  1  0  5  3  1  1  7  6  3  4  3  7  0  1  3  6  3  6  4  6  3  1  7  6  0  7  7  7  3  0  0  3  6  6  7  7  6  3  1  3  0  7  1  0  0  3  7  2  6  7  1  3  1  3  0  0  0  1  3  7  1  3  3  1  7  0  3  6  0  7  3  1  1  7  7  8  3  3  1  3  7  7  1  3  1  0  3  7  0  0  7  7  6  7  6  2  3  3  8  6  3  7  0  8  1  6  6  5  3  8  3  7  8  7  7  6  6  3  8  3  3  0  3  0  7  0  4  8  1  6  5  3  1  3  8  2  1  3  6  6  1  0  1  0  8  3  6  3  7  3  3  0  1  0  6  1  3  4  7  6  3  4  3  7  2  7  3  4  7  3  4  5  6  0  3  6  6  8  3  6  7  3  3  7  7  3  6  1  8  8  7  1  8  1  7  8  0  2  5  2  8  3  1  0  4  1  3  7  8  0  1  7  1  1  0  0  8  0  0  3  4  7  7  7  8  3  7  7  2  3  0  0  6  3  5  7  1  0  3  6  4  3  1  1  3  5  6  0  8  6  1  8  7  2  3  8  6  6  8  6  0  1  7  6  7  3  1  7  3  5  0  7  8  7  6  7  7  1  0  1  0  0  3  7  4  7  4  8  7  0  8  5  3  3  7  4  4  0  0  0  3  3  1  0  7  5  8  3  3  1  5  8  3  7  6  1  4  3  2  3  1  3  1  8  7  0  7  7  3  3  0  3  6  8  3  4  8  3  0  7  6  0  7  3  6  0  4  2  3  8  8  6  6  7  6  3  6  3  7  3  0  3  3  7  0  4  1  8  7  6  6  6  7  7  6  1  8  8  1  8  0  0  4  7  0  6  8  0  7  7  7  7  8  1  3  3  4  0  6  3  7  6  3  7  7  0  4  8  3  0  2  3  6  6  7  3  8  0  6  7  6  3  7  7  7  8  6  7  7  5  7  7  0  0  3  1  7  7  1  1  3  3  8  1  1  6  2  0  3  0  5  0  6  0  1  6  0  7  4  7  4  2  1  8  3  8  0  7  6  7  4  3  1  3  0  3  2  3  8  3  8  7  8  0  7  3  4  8  1  7  6  7  0  3  7  6  3  6  8  4  3  8  4  3  1  6  0  7  3  1  8  8  3  7  3  8  3  7  3  7  7  1  3  2  6  7  3  4  8  3  7  6  0  1  3  8  0  3  7  7  0  7  7  0  4  3  7  7  5  0  8  6  6  7  0  1  8  6  1  6  6  0  7  8  8  7  0  1  1  0  1  8  3  3  6  8  3  3  0  7  7  1  7  0  4  7  3  8  6  3  2  7  7  6  1  3  1  3  0  3  3  7  8  7  8  3  6  0  2  0  7  3  6  6  8  6  7  2  6  7  1  1  6  0  7  8  0  3  3  7  6  7  6  3  7  7  3  0  8  6  8  0  1  1  6  7  0  6  6  3  1  6  6  3  6  0  6  6  1  7  8  7  0  7  7  6  7  0  7  4  0  6  3  3  8  0  3  2  7  1  0  0  3  7  7  7  8  3  8  3  7  1  4  1  7  7  7  7  0  0  6  1  3  1  7  1  1  0  3  7  1  0  6  7  1  8  1  1  0  7  1  5  6  3  3  3  6  7  0  3  0  3  3  3  6  3  3  1  3  6  1  7  8  4  7  2  6  4  4  4  1  8  4  6  5  6  1  3  3  1  0  7  7  4  0  1  6  7  1  3  6  1  6  7  7  8  3  1  7  7  0  2  7  7  3  0  1  3  8  6  3  3  7  8  1  7  1  3  8  0  6  3  0  3  8  7  1  1  8  1  6  4  3  3  8  5  6  1  7  3  7  7  3  7  1  3  3  7  3  6  7  3  1  7  3  7  8  6  3  3  7  7  1  0  0  6  7  3  7  3  6  3  1  7  8  7  3  6  7  3  8  1  8  1  1  1  0  6  6  2  6  0  3  3  3  6  8  3  3  0  0  3  6  3  1  3  8  6  1  6  8  3  0  1  0  1  3  8  3  1  3  3  6  1  8  1  4  0  1  3  3  7  0  0  7  1  6  4  0  7  0  1  3  3  8  7  6  0  8  7  1  0  8  6  7  3  6  0  7  6  3  7  0  6  4  7  7  2  0  3  3  0  7  7  7  8  3  0  3  2  0  7  7  1  1  3  4  7  5  8  0  7  0  7  3  7  7  2  3  7  6  1  0  6  1  6  7  0  8  8  8  1  0  3  6  7  6  7  1  0  3  0  0  6  8  3  7  7  3  6  6  7  0  3  3  0  3  1  3  3  1  8  7  6  0  6  0  7  6  5  1  1  0  3  1  7  3  7  0  1  1  7  6  6  6  7  8  2  3  8  1  7  3  8  8  7  0  3  6  5  1  6  5  7  3  6  7  6  3  1  6  3  7  7  4  0  6  7  7  3  8  3  8  4  7  7  6  3  0  4  1  6  6  8  6  5  8  7  0  7  6  0  0  1  7  6  3  0  8  7  3  1  7  3  1  3  6  0  6  7  1  3  7  1  8  7  0  3  3  3  4  7  6  6  1  8  3  8  0  6  7  1  6  3  1  6  8  4  6  3  7  4  8  0  8  3  8  3  1  1  3  8  8  7  7  8  7  8  6  6  8  3  8  6  7  7  3  4  7  6  3  2  6  3  6  1  8  3  8  7  0  1  6  3  7  6  6  1  2  3  7  1  0  8  7  0  2  3  4  0  0  7  8  6  7  3  7  7  0  8  4  3  8  0  4  6  0  0  6  7  7  7  1  3  7  8  0  4  1  0  6  7  1  2  6  2  1  3  3  8  7  6  4  5  3  0  1  7  3  7  6  0  3  3  7  1  3  3  3  7  1  0  6  3  0  3  7  0  7  7  6  1  7  7  7  8  0  4  3  8  7  3  6  1  6  6  1  3  1  0  3  8  7  1  7  6  2  0  3  6  4  6  3  3  7  8  3  3  3  6  4  6  3  0  2  7  3  8  0  8  8  0  6  6  7  6  7  1  6  1  8  0  7  6  7  0  7  0  1  7  0  3  3  8  1  2  3  0  8  3  3  3  8  7  0  8  4  0  4  7  5  3  6  4  7  7  4  7  3  3  1  8  6  3  3  6  3  2  8  1  7  6  1  6  3  6  6  6  0  1  0  3  1  7  6  6  7  7  0  2  0  0  1  6  1  8  6  6  0  3  8  3  1  0  6  7  3  2  1  1  8  7  7  3  2  0  7  1  8  3  8  8  0  6  8  2  0  8  8  6  8  0  8  7  2  4  8  7  7  4  0  3  3  6  7  0  7  3  7  2  3  7  1  1  3  8  8  6  3  2  7  3  3  7  8  6  7  3  6  6  3  3  7  0  6  7  7  0  3  8  4  3  3  3  1  2  7  6  4  8  1  6  0  3  4  6  1  3  6  7  3  3  3  8  2  7  7  0  1  3  8  7  0  5  7  4  7  3  3  5  0  0  3  3  8  3  3  6  6  7  6  7  7  3  1  3  6  4  1  8  7  3  3  3  7  7  6  1  2  8  0  1  6  7  3  6  1  8  1  7  7  1  8  8  3  6  0  2  4  0  1  6  6  7  1  6  7  6  2  3  1  8  8  0  3  3  3  8  7  7  1  4  6  2  6  3  7  8  6  7  6  3  1  3  0  7  7  7  7  7  3  8  0  7  6  7  6  4  6  3  3  6  1  0  7  6  3  1  8  0  6  6  3  2  8  1  1  0  6  8  0  0  2  7  8  3  8  7  3  6  8  8  7  6  7  3  5  1  1  6  1  7  4  8  3  6  4  7  6  6  3  1  6  3  1  5  7  8  0  6  1  3  4  3  6  7  3  3  3  6  3  8  3  0  1  6  0  0  7  7  8  2  3  6  6  1  3  6  0  8  3  7  1  1  3  1  3  6  3  7  3  3  3  0  7  0  7  7  7  3  6  1  4  7  1  1  6  3  6  6  3  3  1  8  8  7  1  7  0  3  3  7  6  7  0  7  7  0  7  0  7  7  3  6  7  7  8  0  8  6  3  7  0  0  4  3  6  0  7  7  8  0  7  1  7  8  8  3  7  8  7  3  3  3  0  0  3  3  1  7  6  7  5  7  0  0  6  4  0  3  2  7  8  8  0  3  1  0  6  8  3  0  8  5  1  5  4  3  8  7  3  0  8  1  5  6  4  2  8  7  0  3  3  6  2  7  7  6  1  7  0  7  0  3  3  6  3  0  0  6  3  8  5  3  4  7  6  3  3  0  6  3  0  7  1  4  5  3  0  6  1  7  3  4  3  7  0  4  3  8  6  6  3  7  5  3  6  3  7  0  0  3  3  0  1  8  3  0  8  4  3  7  3  3  0  8  3  3  3  3  6  7  6  6  3  6  1  1  3  6  1  8  0  1  6  8  1  7  0  7  7  1  7  3  1  3  0  1  1  6  1  0  7  7  7  7  6  4  3  3  7  0  3  3  1  8  8  8  0  1  8  3  8  7  0  0  6  6  1  3  8  7  0  6  8  2  8  6  7  7  1  0  7  7  1  8  6  3  7  7  3  8  7  0  3  8  3  3  3  3  2  8  6  7  8  3  8  6  3  3  7  3  1  8  1  7  8  7  3  3  7  4  7  7  1  6  3  7  4  1  7'
raw_list = [int(i) for i in raw.split()]
gap_churn_order['gap_cluster'] = raw_list
gap_churn_order['filename'] = gap_churn_order['UNIQID'].str.lstrip('r').astype('int')+1
gap_churn_order = gap_churn_order.drop(columns=['UNIQID'])

churn_merged = pd.merge(churn, gap_churn_order[['gap_cluster', 'filename']], on='filename').drop(columns=['filename'])


In [7]:
churn_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CreditScore        5000 non-null   int64  
 1   Age                5000 non-null   int64  
 2   Tenure             5000 non-null   int64  
 3   Balance            5000 non-null   float64
 4   NumOfProducts      5000 non-null   int64  
 5   HasCrCard          5000 non-null   int64  
 6   IsActiveMember     5000 non-null   int64  
 7   EstimatedSalary    5000 non-null   float64
 8   Exited             5000 non-null   int64  
 9   Geography_Germany  5000 non-null   int64  
 10  Geography_Spain    5000 non-null   int64  
 11  Gender_Male        5000 non-null   int64  
 12  gap_cluster        5000 non-null   int64  
dtypes: float64(2), int64(11)
memory usage: 507.9 KB


In [8]:

# X = 特徵, y = 目標
X = churn_merged.drop(columns=['Exited'])
y = churn_merged['Exited']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [9]:
# === 1) Naive ===

ols = LinearRegression().fit(X_train, y_train)
y_score = ols.predict(X_test)
y_pred_ols = (y_score >= 0.5).astype(int)

print("\n=== Linear (OLS threshold 0.5) ===")
print(confusion_matrix(y_test, y_pred_ols))
print(classification_report(y_test, y_pred_ols, digits=4))



=== Linear (OLS threshold 0.5) ===
[[751 149]
 [266 334]]
              precision    recall  f1-score   support

           0     0.7384    0.8344    0.7835       900
           1     0.6915    0.5567    0.6168       600

    accuracy                         0.7233      1500
   macro avg     0.7150    0.6956    0.7002      1500
weighted avg     0.7197    0.7233    0.7168      1500



In [10]:
# === 2) Linear SVC ===
lsvc = LinearSVC(random_state=42, max_iter=1000, class_weight='balanced')
model3 = lsvc.fit(X_train, y_train)
y_pred3 = model3.predict(X_test)
print("\n=== Linear SVC ===")
print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred3))
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred3, digits=4))


=== Linear SVC ===
=== Confusion Matrix ===
[[587 313]
 [186 414]]

=== Classification Report ===
              precision    recall  f1-score   support

           0     0.7594    0.6522    0.7017       900
           1     0.5695    0.6900    0.6240       600

    accuracy                         0.6673      1500
   macro avg     0.6644    0.6711    0.6628      1500
weighted avg     0.6834    0.6673    0.6706      1500



In [33]:
# === 3) Logistic Regression ===
lr = LogisticRegression(max_iter=100, random_state=42, solver='liblinear', class_weight='balanced')
model4 = lr.fit(X_train, y_train)
y_pred4 = model4.predict(X_test)
print("\n=== Logistic Regression ===")
print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred4))
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred4, digits=4))


=== Logistic Regression ===
=== Confusion Matrix ===
[[589 311]
 [184 416]]

=== Classification Report ===
              precision    recall  f1-score   support

           0     0.7620    0.6544    0.7041       900
           1     0.5722    0.6933    0.6270       600

    accuracy                         0.6700      1500
   macro avg     0.6671    0.6739    0.6656      1500
weighted avg     0.6861    0.6700    0.6733      1500



In [34]:
# === 4) RandomForestClassifier ===
rfc = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=2, max_leaf_nodes=2, n_jobs=-1, class_weight='balanced')
model1 = rfc.fit(X_train, y_train)
y_pred = model1.predict(X_test)
print("\n=== RandomForestClassifier ===")
print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, digits=4))


=== RandomForestClassifier ===
=== Confusion Matrix ===
[[693 207]
 [205 395]]

=== Classification Report ===
              precision    recall  f1-score   support

           0     0.7717    0.7700    0.7709       900
           1     0.6561    0.6583    0.6572       600

    accuracy                         0.7253      1500
   macro avg     0.7139    0.7142    0.7140      1500
weighted avg     0.7255    0.7253    0.7254      1500



In [35]:
# === 5) XGBoost ===
xgb = XGBClassifier(n_estimators=100, max_depth=2, max_leaves=2, learning_rate=0.1, random_state=42, n_jobs=-1, eval_metric='logloss', tree_method='hist')
model5 = xgb.fit(X_train, y_train)
y_pred5 = model5.predict(X_test)
print("\n=== XGBoost ===")
print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred5))
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred5, digits=4))


=== XGBoost ===
=== Confusion Matrix ===
[[792 108]
 [222 378]]

=== Classification Report ===
              precision    recall  f1-score   support

           0     0.7811    0.8800    0.8276       900
           1     0.7778    0.6300    0.6961       600

    accuracy                         0.7800      1500
   macro avg     0.7794    0.7550    0.7619      1500
weighted avg     0.7798    0.7800    0.7750      1500



In [36]:
# # === 6) LightGBM ===
lgbm = LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=2, num_leaves=2, random_state=42, n_jobs=-1, class_weight='balanced')
model6 = lgbm.fit(X_train, y_train)
y_pred6 = model6.predict(X_test)
print("\n=== LightGBM ===")
print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred6))
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred6, digits=4))


=== LightGBM ===
=== Confusion Matrix ===
[[713 187]
 [157 443]]

=== Classification Report ===
              precision    recall  f1-score   support

           0     0.8195    0.7922    0.8056       900
           1     0.7032    0.7383    0.7203       600

    accuracy                         0.7707      1500
   macro avg     0.7614    0.7653    0.7630      1500
weighted avg     0.7730    0.7707    0.7715      1500



---
---

In [ ]:


y_col = 'Exited'
c_col = 'gap_cluster'
X_cols = [c for c in churn_merged.columns if c not in (y_col, c_col)]
y = churn_merged[y_col]
c = churn_merged[c_col]

strat_key = y.astype(str) + '|' + c.astype(str)
X_train, X_test, y_train, y_test, c_train, c_test = train_test_split(churn_merged[X_cols], y, c, test_size=0.3, stratify=strat_key, random_state=42)

labels_all = np.sort(y.unique())
rows = []
confusion_matrix_= {}
classification_report_ = {}
for k in sorted(c_train.unique()):
    train_mask = (c_train == k)
    test_mask = (c_test == k)
    Xk_train, yk_train = X_train[train_mask], y_train[train_mask]
    Xk_test, yk_test = X_test[test_mask], y_test[test_mask]

    if len(yk_train.unique())<2 or len(yk_test)==0:
        rows.append({'cluster':k, 'n_train':len(yk_train), 'n_test':len(yk_test), 'Acc':np.nan, 'F1':np.nan, 'note':'類別不足或無測試樣本'})
        continue

    naive_ = DummyClassifier(strategy='prior', random_state=42)
    naive_.fit(Xk_train, yk_train)
    naive__y_hat = naive_.predict(Xk_test)
    naive__y_proba = naive_.predict_proba(Xk_test)[:,1]
    rows.append({'cluster':k, 'model':'naive', 'n_train':len(yk_train), 'n_test':len(yk_test), 'Acc':accuracy_score(yk_test, naive__y_hat), 'F1':f1_score(yk_test, naive__y_hat, average='macro'), 'AUC':roc_auc_score(yk_test, naive__y_proba)})
    confusion_matrix_[(k, 'naive')] = confusion_matrix(yk_test, naive__y_hat, labels=labels_all)
    classification_report_[(k, 'naive')] = classification_report(yk_test, naive__y_hat)

    lsvm = LinearSVC(random_state=42, max_iter=1000, class_weight='balanced')
    lsvm.fit(Xk_train, yk_train)
    lsvm__y_hat = lsvm.predict(Xk_test)
    lsvm__y_prob = lsvm.decision_function(Xk_test)
    rows.append({'cluster':k, 'model':'lsvm', 'n_train':len(yk_train), 'n_test':len(yk_test), 'Acc':accuracy_score(yk_test, lsvm__y_hat), 'F1':f1_score(yk_test, lsvm__y_hat, average='macro'), 'AUC':roc_auc_score(yk_test, lsvm__y_prob)})
    confusion_matrix_[(k, 'LinearSVC')] = confusion_matrix(yk_test, lsvm__y_hat, labels=labels_all)
    classification_report_[(k, 'LinearSVC')] = classification_report(yk_test, lsvm__y_hat)

    lr = LogisticRegression(max_iter=100, random_state=42, solver='liblinear', class_weight='balanced')
    lr.fit(Xk_train, yk_train)
    lr_y_hat = lr.predict(Xk_test)
    lr__y_proba = lr.predict_proba(Xk_test)[:,1]
    rows.append({'cluster':k, 'model':'lr', 'n_train':len(yk_train), 'n_test':len(yk_test), 'Acc':accuracy_score(yk_test, lr_y_hat), 'F1':f1_score(yk_test, lr_y_hat, average='macro'), 'AUC':roc_auc_score(yk_test, lr__y_proba)})
    confusion_matrix_[(k, 'LogisticRegression')] = confusion_matrix(yk_test, lr_y_hat, labels=labels_all)
    classification_report_[(k, 'LogisticRegression')] = classification_report(yk_test, lr_y_hat) 
    rows.append({'cluster':k, 'model':'rfc', 'n_train':len(yk_train), 'n_test':len(yk_test), 'Acc':accuracy_score(yk_test, rfc_y_hat), 'F1':f1_score(yk_test, rfc_y_hat, average='macro'), 'AUC':roc_auc_score(yk_test, rfc__y_proba)})
    confusion_matrix_[(k, 'RandomForestClassifier')] = confusion_matrix(yk_test, rfc_y_hat, labels=labels_all)
    classification_report_[(k, 'RandomForestClassifier')] = classification_report(yk_test, rfc_y_hat)

    xgbc = XGBClassifier(n_estimators=100, max_depth=2, max_leaves=2, learning_rate=0.1, random_state=42, n_jobs=-1, eval_metric='logloss', tree_method='hist')
    xgbc.fit(Xk_train, yk_train)
    xgbc_y_hat = xgbc.predict(Xk_test)
    xgbc__y_proba = xgbc.predict_proba(Xk_test)[:,1]
    rows.append({'cluster':k, 'model':'xgbc', 'n_train':len(yk_train), 'n_test':len(yk_test), 'Acc':accuracy_score(yk_test, xgbc_y_hat), 'F1':f1_score(yk_test, xgbc_y_hat, average='macro'), 'AUC':roc_auc_score(yk_test, xgbc__y_proba)})
    confusion_matrix_[(k, 'XGBClassifier')] = confusion_matrix(yk_test, xgbc_y_hat, labels=labels_all)
    classification_report_[(k, 'XGBClassifier')] = classification_report(yk_test, xgbc_y_hat)

    lgbmc = LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=2, num_leaves=2, random_state=42, n_jobs=-1, class_weight='balanced', verbose=-1)
    lgbmc.fit(Xk_train, yk_train)
    lgbmc_y_hat = lgbmc.predict(Xk_test)
    lgbmc__y_proba = lgbmc.predict_proba(Xk_test)[:,1]
    rows.append({'cluster':k, 'model':'lgbmc', 'n_train':len(yk_train), 'n_test':len(yk_test), 'Acc':accuracy_score(yk_test, lgbmc_y_hat), 'F1':f1_score(yk_test, lgbmc_y_hat, average='macro'), 'AUC':roc_auc_score(yk_test, lgbmc__y_proba)})
    confusion_matrix_[(k, 'LGBMClassifier')] = confusion_matrix(yk_test, lgbmc_y_hat, labels=labels_all)
    classification_report_[(k, 'LGBMClassifier')] = classification_report(yk_test, lgbmc_y_hat)

report = pd.DataFrame(rows).sort_values(['cluster','model'])
report

c:\Users\USER\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\USER\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\USER\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\USER\anaconda3\Lib\sit

,cluster,model,n_train,n_test,Acc,F1,AUC
5,0,lgbmc,590,253,0.715415,0.710490,0.830634
2,0,lr,590,253,0.652174,0.649761,0.708376
1,0,lsvm,590,253,0.644269,0.642251,0.710764
0,0,naive,590,253,0.588933,0.370647,0.500000
3,0,rfc,590,253,0.723320,0.718531,0.796819
4,0,xgbc,590,253,0.747036,0.733509,0.837119
11,1,lgbmc,585,252,0.765873,0.760128,0.843683
8,1,lr,585,252,0.642857,0.640593,0.708412
7,1,lsvm,585,252,0.646825,0.644356,0.708868
6,1,naive,585,252,0.591270,0.371571,0.500000


---
---

In [ ]:
print(classification_report_[(2, 'naive')])

              precision    recall  f1-score   support

           0       0.63      1.00      0.78        93
           1       0.00      0.00      0.00        54

    accuracy                           0.63       147
   macro avg       0.32      0.50      0.39       147
weighted avg       0.40      0.63      0.49       147



In [50]:
print(classification_report_[(5, 'naive')])

              precision    recall  f1-score   support

           0       0.52      1.00      0.69        11
           1       0.00      0.00      0.00        10

    accuracy                           0.52        21
   macro avg       0.26      0.50      0.34        21
weighted avg       0.27      0.52      0.36        21



In [51]:
print(classification_report_[(2, 'RandomForestClassifier')])

              precision    recall  f1-score   support

           0       0.70      0.86      0.78        22
           1       0.67      0.43      0.52        14

    accuracy                           0.69        36
   macro avg       0.69      0.65      0.65        36
weighted avg       0.69      0.69      0.68        36



In [52]:
print(classification_report_[(5, 'RandomForestClassifier')])

              precision    recall  f1-score   support

           0       0.83      0.91      0.87        11
           1       0.89      0.80      0.84        10

    accuracy                           0.86        21
   macro avg       0.86      0.85      0.86        21
weighted avg       0.86      0.86      0.86        21



In [39]:
print(classification_report_[(2, 'XGBClassifier')])

              precision    recall  f1-score   support

           0       0.76      0.86      0.81        22
           1       0.73      0.57      0.64        14

    accuracy                           0.75        36
   macro avg       0.74      0.72      0.72        36
weighted avg       0.75      0.75      0.74        36



In [40]:
print(classification_report_[(5, 'XGBClassifier')])

              precision    recall  f1-score   support

           0       0.75      0.82      0.78        11
           1       0.78      0.70      0.74        10

    accuracy                           0.76        21
   macro avg       0.76      0.76      0.76        21
weighted avg       0.76      0.76      0.76        21



In [41]:
print(classification_report_[(8, 'XGBClassifier')])

              precision    recall  f1-score   support

           0       0.77      0.85      0.81        93
           1       0.68      0.56      0.61        54

    accuracy                           0.74       147
   macro avg       0.72      0.70      0.71       147
weighted avg       0.74      0.74      0.73       147



In [42]:
print(classification_report_[(8, 'LGBMClassifier')])

              precision    recall  f1-score   support

           0       0.78      0.74      0.76        93
           1       0.59      0.63      0.61        54

    accuracy                           0.70       147
   macro avg       0.68      0.69      0.68       147
weighted avg       0.71      0.70      0.70       147



In [43]:
print(classification_report_[(2, 'LinearSVC')])

              precision    recall  f1-score   support

           0       0.76      0.59      0.67        22
           1       0.53      0.71      0.61        14

    accuracy                           0.64        36
   macro avg       0.65      0.65      0.64        36
weighted avg       0.67      0.64      0.64        36



---
---

In [44]:
import pandas as pd

In [45]:
data = pd.read_csv(r'C:\Users\USER\Documents\GitHub\psychic-spoon\DataSet\secom.csv', sep='\t').drop(columns=['Time'])
data.head()

,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,x11,x12,x13,x14,x15,x16,x17,x18,x19,x20,x21,x22,x23,x24,x25,x26,x27,x28,x29,x30,x31,x32,x33,x34,x35,x36,x37,x38,x39,x40,...,x552,x553,x554,x555,x556,x557,x558,x559,x560,x561,x562,x563,x564,x565,x566,x567,x568,x569,x570,x571,x572,x573,x574,x575,x576,x577,x578,x579,x580,x581,x582,x583,x584,x585,x586,x587,x588,x589,x590,Pass/Fail
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,-0.0034,0.9455,202.4396,0.0,7.9558,414.8710,10.0433,0.9680,192.3963,12.5190,1.4026,-5419.00,2916.50,-4043.75,751.00,0.8955,1.7730,3.0490,64.2333,2.0222,0.1632,3.5191,83.3971,9.5126,50.6170,64.2588,49.3830,66.3141,86.9555,117.5132,...,0.78,0.1827,5.7349,0.3363,39.8842,3.2687,1.0297,1.0344,0.4385,0.1039,42.3877,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,533.8500,2.1113,8.95,0.3157,3.0624,0.1026,1.6765,14.9509,NaN,NaN,NaN,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,-0.0148,0.9627,200.5470,0.0,10.1548,414.7347,9.2599,0.9701,191.2872,12.4608,1.3825,-5441.50,2604.25,-3498.75,-1640.25,1.2973,2.0143,7.3900,68.4222,2.2667,0.2102,3.4171,84.9052,9.7997,50.6596,64.2828,49.3404,64.9193,87.5241,118.1188,...,1.33,0.2829,7.1196,0.4989,53.1836,3.9139,1.7819,0.9634,0.1745,0.0375,18.1087,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,535.0164,2.4335,5.92,0.2653,2.0111,0.0772,1.1065,10.9003,0.0096,0.0201,0.0060,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,0.0013,0.9615,202.0179,0.0,9.5157,416.7075,9.3144,0.9674,192.7035,12.5404,1.4123,-5447.75,2701.75,-4047.00,-1916.50,1.3122,2.0295,7.5788,67.1333,2.3333,0.1734,3.5986,84.7569,8.6590,50.1530,64.1114,49.8470,65.8389,84.7327,118.6128,...,0.85,0.0857,7.1619,0.3752,23.0713,3.9306,1.1386,1.5021,0.3718,0.1233,24.7524,267.064,0.9032,1.10,0.6219,0.4122,0.2562,0.4119,68.8489,535.0245,2.0293,11.21,0.1882,4.0923,0.0640,2.0952,9.2721,0.0584,0.0484,0.0148,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,-0.0033,0.9629,201.8482,0.0,9.6052,422.2894,9.6924,0.9687,192.1557,12.4782,1.4011,-5468.25,2648.25,-4515.00,-1657.25,1.3137,2.0038,7.3145,62.9333,2.6444,0.2071,3.3813,84.9105,8.6789,50.5100,64.1125,49.4900,65.1951,86.6867,117.0442,...,39.33,0.6812,56.9303,17.4781,161.4081,35.3198,54.2917,1.1613,0.7288,0.2710,62.7572,268.228,0.6511,7.32,0.1630,3.5611,0.0670,2.7290,25.0363,530.5682,2.0253,9.33,0.1738,2.8971,0.0525,1.7585,8.5831,0.0202,0.0149,0.0044,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,-0.0072,0.9569,201.9424,0.0,10.5661,420.5925,10.3387,0.9735,191.6037,12.4735,1.3888,-5476.25,2635.25,-3987.50,117.00,1.2887,1.9912,7.2748,62.8333,3.1556,0.2696,3.2728,86.3269,8.7677,50.2480,64.1511,49.7520,66.1542,86.1468,121.4364,...,1.98,0.4287,9.7608,0.8311,70.9706,4.9086,2.5014,0.9778,0.2156,0.0461,22.0500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,532.0155,2.0275,8.83,0.2224,3.1776,0.0706,1.6597,10.9698,NaN,NaN,NaN,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1


In [46]:
cols = data.columns.to_list()
for col in cols:
    data[col] = data[col].fillna(data[col].mean(axis=0))
data.isna().sum()

x1           0
x2           0
x3           0
x4           0
x5           0
            ..
x587         0
x588         0
x589         0
x590         0
Pass/Fail    0
Length: 591, dtype: int64

In [47]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
# X = 特徵, y = 目標
X = data.drop(columns=['Pass/Fail'])
y = data['Pass/Fail']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

lgbmc = LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=2, num_leaves=2, random_state=42, class_weight='balanced', n_jobs=-1, verbose=-1)
model = lgbmc.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

print(accuracy_score(y_pred, y_test))
print(classification_report(y_pred, y_test))

0.8492569002123143
              precision    recall  f1-score   support

          -1       0.88      0.96      0.92       401
           1       0.48      0.21      0.30        70

    accuracy                           0.85       471
   macro avg       0.68      0.59      0.61       471
weighted avg       0.82      0.85      0.82       471

